# oracle13_run — chấm TRỌN mọi đoạn của 13 văn bản gold đang thua. **1,7 phút GPU.**

**Câu hỏi đã viết ra trước:** trong 13 câu E đang sai mà gold vẫn nằm trong rổ 50, có bao
nhiêu câu **tồn tại một đoạn** mà chính model hiện tại chấm đủ cao để lọt top-5?

- **Có đoạn đó** → lỗi ở khâu **CHỌN ĐOẠN / cổng M**, sửa được, **0 tham số phát sinh**.
- **Không có** → lỗi ở **MODEL**, và trong ngân sách 3B thì E hết đường — dồn sức cho D.

Đây là phép đo rẻ nhất còn lại mà vẫn quyết định được hướng đi. Chốt xong thì hết đoán.

### Vì sao phải chạy lại chứ không đọc `oracle_chunks_dev300.json` cũ

Bản 17/08 chạy trên **rổ BM25 cũ**, **24 văn bản khác**, và kết luận "chọn đoạn chỉ còn
+0,67". Rổ đã đổi sang fusion, tập câu sai đã đổi hẳn. Bài học chính file này ghi:
*"đừng ngoại suy chẩn đoán từ rổ cũ sang rổ mới"* — lần trước ước 5–6 câu "cả rổ chấm ~0",
thực tế còn 1.

### ⚠️ PHÁT HIỆN 23/08 khiến lượt này cần thiết hơn: **5/14 văn bản gold CHƯA HỀ ĐƯỢC ĐỌC SÂU**

`deepen_all` chỉ đọc sâu **M=20 văn bản đứng đầu theo `ce` tầng 1**. Năm văn bản gold rớt
ngoài mốc đó nên **chưa bao giờ được chấm bằng khâu đọc sâu**:

| qid | gold | hạng tầng 1 | `ce` | `ce_deep` |
|---|---|---|---|---|
| 49070 | 129275 | **42** | 0,00017 | — chưa đọc |
| 16942 | 102434 | **40** | 0,00023 | — chưa đọc |
| 65498 | 285041 | **40** | 0,01057 | — chưa đọc |
| 64622 | 149370 | **28** | 0,00398 | — chưa đọc |
| 63832 | 213302 | **21** | 0,00112 | — chưa đọc |

Điểm ~0 của chúng là chấm trên **3 đoạn của D**, không phải phán quyết của khâu đọc sâu.
**Gọi chúng là "model mù" là sai** — chưa ai cho model cơ hội đọc.

Và khi được đọc sâu thì gold lên rất mạnh: `65498/23402` **0,163 → 0,862** · `8946`
0,00045 → 0,013 (29×) · `92466` 0,029 → 0,123 (4×). Nên câu hỏi "nếu M lớn hơn thì sao"
là câu hỏi thật, chưa ai trả lời.

### Ba con số lượt này sinh ra

1. **Trần chọn đoạn** — bao nhiêu câu thắng nếu gold được lấy điểm đoạn TỐT NHẤT của nó.
2. **Cổng M có phải nút thắt không** — riêng 5 văn bản chưa đọc sâu, đọc rồi có cứu được?
3. **Danh sách câu vô vọng** — gold có đoạn tốt nhất vẫn thua. Đó mới là trần thật của model.

### ⚠️ Cách đọc: đây là ORACLE, không phải điểm chạy được

Chỉ gold được nâng, đối thủ giữ nguyên điểm cũ. Lượt oracle 17/08 dùng **hệ số 0,6** để
quy về mức thực tế — giữ đúng quy ước đó, notebook in cả hai con số. **Số oracle là chặn
TRÊN**, đừng mang đi hứa hẹn.

**Upload:** `Ketqua_E/scores_dev300_fusion_M20_K20.json` (nếu chưa). Còn lại đã có sẵn.
**Chi phí:** 1.353 đoạn ≈ **1,7 phút** trên GPU.

> 💡 **KHÔNG CẦN GPU.** Notebook tự dò: có GPU thì dùng, không thì chạy CPU (~20 phút).
> Hàng đợi GPU của Kaggle hay nghẽn — với job nhỏ thế này, chọn **Accelerator = None**
> rồi chạy ngay còn nhanh hơn ngồi đợi tới lượt. Kết quả y hệt, chỉ khác thời gian.


In [ ]:
!pip install -q sentence-transformers

In [ ]:
# ===== Bước 1: tìm 13 câu đang thua, liệt kê mọi đoạn của gold =====
import os, sys, json, time, hashlib
import torch

# Lượt này nhỏ tới mức KHÔNG CẦN GPU. 1.353 đoạn, model 0,568B.
# Hàng đợi GPU của Kaggle hay nghẽn ("position in the queue is 5"), hàng đợi CPU thì không.
# Chạy CPU chậm hơn ~15 lần nhưng vẫn chỉ ~20 phút, và bắt đầu NGAY.
DEV = "cuda" if torch.cuda.is_available() else "cpu"
NHIP = 12.9 if DEV == "cuda" else 1.2          # đoạn/giây, đo thật trên T4 / ước cho CPU 4 nhân
if DEV == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    torch.set_num_threads(os.cpu_count() or 4)
    print(f"KHÔNG có GPU -> chạy CPU trên {os.cpu_count()} nhân. Chậm hơn nhưng không phải xếp hàng.")

MOC, TOPK, HESO = 0.9350, 5, 0.6      # HESO: quy ước quy oracle về thực tế, đặt 17/08

INPUT_DIR = next(p for p in ("/kaggle/input/project-ir",
                             "/kaggle/input/datasets/locdovan211/project-ir")
                 if os.path.isdir(p))
CTX_DIR = next(p for p in (f"{INPUT_DIR}/selected-contexts/selected-contexts",
                           f"{INPUT_DIR}/selected-contexts")
               if os.path.isdir(p) and any(f.startswith("context_") for f in os.listdir(p)))
OUT = "/kaggle/working/outputs"; os.makedirs(OUT, exist_ok=True)
sys.path.append(INPUT_DIR)

import deep_chunk as DC
from rerank import load_reranker
from rerank_from_d import blend_bm25_first
DC.MERGE_CHARS = 1800                 # BẮT BUỘC: quên dòng này thì đếm ra đoạn THÔ, sai 2x
assert DC.MERGE_CHARS == 1800

S    = json.load(open(f"{INPUT_DIR}/scores_dev300_fusion_M20_K20.json", encoding="utf-8"))
dev  = json.load(open(f"{INPUT_DIR}/dev_300_locked.json", encoding="utf-8"))
cand = json.load(open(f"{INPUT_DIR}/fusion_rrf_top50_dev_1000.json", encoding="utf-8"))
qids = [q for q in dev if q in S]
GOLD = {q: {str(a) for a in dev[q]["answer"]} for q in qids}
ORDER= {q: [str(c["doc_id"]) for c in
            sorted(cand[q], key=lambda c: -float(c["rrf_score"]))] for q in qids}
CE   = lambda v: max(v.get("ce", -9), v.get("ce_deep", -9))
rec  = lambda p: sum(len(GOLD[q] & set(p[q])) / len(GOLD[q]) for q in qids) / len(qids)

PIPE = {q: blend_bm25_first(DC.rank_by(S[q], "max"), ORDER[q], k=TOPK, n_bm25=1) for q in qids}
assert abs(rec(PIPE) - MOC) < 1e-3, f"KHÔNG dựng lại được mốc {MOC} -> sai chỗ nạp, DỪNG"
bad = [q for q in qids if not (GOLD[q] & set(PIPE[q]))]
inb = [q for q in bad if GOLD[q] & set(S[q])]
print(f"\nmốc {rec(PIPE):.4f} · sai {len(bad)} câu · trong đó {len(inb)} câu gold còn trong rổ")
assert len(inb) == 13, f"chờ 13 câu, thấy {len(inb)} -> chẩn đoán 23/08 đã lệch, soi lại"

jobs = []                              # (qid, doc_id, đoạn)
for q in inb:
    for d in sorted(GOLD[q] & set(S[q])):
        for c in DC.doc_chunks(CTX_DIR, d):
            jobs.append((q, d, c))
print(f"{len(jobs):,} đoạn phải chấm -> ước {len(jobs)/NHIP/60:.0f} phút trên {DEV.upper()}")
assert len(jobs) < 6000, "nhiều bất thường — kiểm MERGE_CHARS"

In [ ]:
# ===== Bước 2: chấm TOÀN BỘ đoạn (không chọn lọc gì cả) =====
score_fn = load_reranker("AITeamVN/Vietnamese_Reranker", device=DEV, max_length=1024)
t0 = time.time()
sc = score_fn.predict([[dev[q]["question"], c] for q, d, c in jobs])
print(f"xong {(time.time()-t0)/60:.1f} phút · {len(jobs)/(time.time()-t0):.1f} đoạn/s")

BEST = {}                              # (qid, doc) -> điểm đoạn cao nhất
for (q, d, _), v in zip(jobs, sc):
    k = (q, d)
    BEST[k] = max(BEST.get(k, -9.0), float(v))
json.dump({f"{q}|{d}": v for (q, d), v in BEST.items()},
          open(f"{OUT}/oracle13_dev300_fusion.json", "w", encoding="utf-8"), ensure_ascii=False)
print(f"ĐÃ LƯU {OUT}/oracle13_dev300_fusion.json — TẢI VỀ (quy tắc 2)")

In [ ]:
# ===== Bước 3: đọc kết quả =====
print(f"{'qid':>8}{'gold':>9}{'điểm nay':>10}{'ORACLE':>9}{'ngưỡng':>9}{'đã đọc sâu':>12}  phán")
cuu, thua, do_M = [], [], []
for q in inb:
    thr = sorted((CE(v) for v in S[q].values()), reverse=True)[TOPK - 1]
    for d in sorted(GOLD[q] & set(S[q])):
        nay, orc = CE(S[q][d]), BEST[(q, d)]
        sau = "CÓ" if "ce_deep" in S[q][d] else "CHƯA"
        win = orc > thr
        print(f"{q:>8}{d:>9}{nay:>10.5f}{orc:>9.5f}{thr:>9.4f}{sau:>12}  "
              f"{'CỨU ĐƯỢC' if win else 'vẫn thua'}")
        if win:
            cuu.append(q)
            if sau == "CHƯA": do_M.append(q)          # cứu được CHỈ vì được đọc sâu
        else:
            thua.append(q)

ORC = {q: {d: dict(v) for d, v in S[q].items()} for q in qids}
for (q, d), v in BEST.items():
    ORC[q][d]["ce_deep"] = max(v, ORC[q][d].get("ce_deep", -9))
got = rec({q: blend_bm25_first(DC.rank_by(ORC[q], "max"), ORDER[q], k=TOPK, n_bm25=1) for q in qids})

print("\n" + "=" * 66)
print(f"CỨU ĐƯỢC {len(set(cuu))}/13 câu · vẫn thua {len(set(thua)-set(cuu))} câu")
print(f"  trong đó cứu được nhờ ĐƯỢC ĐỌC SÂU (trước rớt ngoài M=20): {len(set(do_M))}  {sorted(set(do_M))}")
print(f"  câu VÔ VỌNG (đoạn tốt nhất vẫn thua) : {sorted(set(thua)-set(cuu))}")
print(f"\nrecall@5 oracle = {got:.4f}  (mốc {MOC})  ->  trần {(got-MOC)*100:+.2f} điểm")
print(f"quy về thực tế × {HESO} = {(got-MOC)*100*HESO:+.2f} điểm")
print("=" * 66)
d = (got - MOC) * 100 * HESO
print("CHỌN ĐOẠN/CỔNG M là nút thắt -> sửa được mà KHÔNG tốn tham số nào. Làm tiếp." if d >= 2.0 else
      "dư địa có nhưng dưới ngưỡng đo được của dev300 -> đừng đổi pipeline vì nó" if d >= 0.7 else
      "MODEL là nút thắt, không phải cách băm đoạn. Trong 3B thì E hết đường -> dồn cho D.")
print("Nhớ: chỉ gold được nâng, đối thủ giữ nguyên -> đây là CHẶN TRÊN, không phải điểm chạy được.")